# VibeSeq Studio — Colab T4
Select a T4 GPU runtime. This notebook serves the built Studio and inference API from the same origin. Both real providers are fixed to medium: `stabilityai/stable-audio-3-medium@27b5a21b791b1b033d193a9e1e3ce78493f102f9` and `MuScriptor/muscriptor-medium@f32236969308476e01fd3aae67357de5feb05a2d`. Small-model and demo fallback are forbidden for real jobs. Standard FlashAttention 2 does not support Turing, so T4 generation is exposed only as the explicit `cuda-t4-sdpa` / `pytorch-sdpa` provisional route. It is disabled by default and is not production-validated.

In [ ]:
from pathlib import Path
import json
import os
import signal
import subprocess
import sys
import time
from urllib.error import URLError
from urllib.request import urlopen

repo = Path(os.environ.get("VIBESEQ_REPO", Path.cwd())).expanduser().resolve()
assert (repo / "server" / "pyproject.toml").is_file(), (
    "Set VIBESEQ_REPO to the VibeSeq checkout"
)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
subprocess.check_call(["nvidia-smi"])

Authenticate only after accepting [Stable Audio 3 medium](https://huggingface.co/stabilityai/stable-audio-3-medium) and [MuScriptor medium](https://huggingface.co/MuScriptor/muscriptor-medium). Add `HF_TOKEN` to Colab Secrets and grant this notebook access. The cell uses interactive login only when neither the process environment nor Colab Secrets provides it, and never prints the token. MuScriptor weights are CC BY-NC 4.0.

In [ ]:
sys.path.insert(0, str(repo / "colab"))
from auth_bootstrap import load_hf_token_from_colab_secrets

if not load_hf_token_from_colab_secrets():
    subprocess.check_call(
        [
            "uv",
            "run",
            "--project",
            str(repo / "server"),
            "--with",
            "huggingface-hub",
            "hf",
            "auth",
            "login",
        ]
    )

In [ ]:
# Leave False until the real T4 SDPA/chunked route is intentionally tested.
ENABLE_PROVISIONAL_T4 = False
PORT = 8000

subprocess.check_call(
    ["uv", "sync", "--frozen", "--project", str(repo / "server"), "--extra", "models"]
)
launch = [
    sys.executable,
    str(repo / "colab" / "run_studio.py"),
    "--repo",
    str(repo),
    "--skip-sync",
    "--port",
    str(PORT),
]
if ENABLE_PROVISIONAL_T4:
    launch.append("--enable-provisional-t4")

previous = globals().get("_vibeseq_studio_process")
if previous is not None and previous.poll() is None:
    os.killpg(previous.pid, signal.SIGTERM)
    previous.wait(timeout=15)

log_path = repo / ".vibeseq-colab.log"
_vibeseq_studio_log = log_path.open("ab", buffering=0)
_vibeseq_studio_process = subprocess.Popen(
    launch,
    cwd=repo,
    stdout=_vibeseq_studio_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

health = None
deadline = time.monotonic() + 900
while time.monotonic() < deadline:
    return_code = _vibeseq_studio_process.poll()
    if return_code is not None:
        raise RuntimeError(
            f"VibeSeq Studio exited with code {return_code}; inspect {log_path.name}."
        )
    try:
        with urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=2) as response:
            health = json.load(response)
        break
    except (URLError, TimeoutError, json.JSONDecodeError):
        time.sleep(1)
if health is None:
    os.killpg(_vibeseq_studio_process.pid, signal.SIGTERM)
    raise RuntimeError(
        f"VibeSeq Studio did not become healthy; inspect {log_path.name}."
    )

for capability_name in ("generation", "transcription"):
    bootstrap = health.get(capability_name, {}).get("bootstrap", {})
    if bootstrap.get("kind") != "huggingface-files":
        raise RuntimeError(f"{capability_name} did not expose an exact file manifest.")
    subprocess.check_call(
        [
            "uv",
            "run",
            "--frozen",
            "--project",
            str(repo / "server"),
            "--with",
            "huggingface-hub",
            "hf",
            "download",
            bootstrap["modelId"],
            "--revision",
            bootstrap["revision"],
            *bootstrap["files"],
        ]
    )
with urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=10) as response:
    health = json.load(response)

safe_health = {
    "target": health.get("target"),
    "hardware": {
        "cudaName": health.get("hardware", {}).get("cudaName"),
        "cudaCapability": health.get("hardware", {}).get("cudaCapability"),
    },
    "generation": {
        key: health.get("generation", {}).get(key)
        for key in (
            "provider",
            "model",
            "route",
            "runtime",
            "ready",
            "provisional",
            "executionEnabled",
        )
    },
    "transcription": {
        key: health.get("transcription", {}).get(key)
        for key in ("provider", "model", "route", "runtime", "ready")
    },
}
print(json.dumps(safe_health, indent=2))

Open the complete Studio through Colab's authenticated kernel-port proxy. If the new window is blocked by the browser, allow pop-ups for Colab and run this cell again. The temporary runtime and its project data disappear when Colab ends. Before stopping the runtime, use Project → Export project bundle and save the .vibeseq file outside Colab; desktop Project → Import project bundle validates and restores the editable workspace. The transfer UI is implemented, but a real T4-to-desktop round trip still remains part of the production gate.

In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(PORT)